# Phase 4: Causal Patching — Activation Patching

**Experimental protocol (Denoising):**
1. Run **clean prompt** → save clean hidden states for all layers (at the last token of `<hop1>`)
2. Run **counterfactual (corrupted) prompt** normally → save the baseline (hallucinated) answer
3. For **each layer**: Run **counterfactual prompt again**, but REPLACE the layer's hidden state (at the last token of `<hop1>`) with the CLEAN state harvested in step 1
4. Measure whether injecting the clean state successfully restores the correct answer

If the correct answer is restored in ≥50% of cases for a given layer, that layer is causally responsible for the hop-1 reasoning step.

In [ ]:
!pip install -q transformers accelerate huggingface_hub
import huggingface_hub
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("Logged in via Kaggle Secrets!")
except Exception as e:
    print(f"Kaggle Secrets failed ({e}), using fallback...")
    hf_token = "REPLACE_ME_WITH_YOUR_RAW_HF_TOKEN"
    huggingface_hub.login(token=hf_token)

In [ ]:
import torch
import numpy as np
import random

# Set seed for determinism
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

import json, torch
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print(f"Model loaded. Device: {model.device}")

In [ ]:
print("Downloading dataset...")
file_path = hf_hub_download(
    repo_id="AnishRacherla/LinguaFranca-Phase3",
    filename="data/2wikimultihopqa/augmented.jsonl",
    repo_type="dataset"
)
with open(file_path) as f:
    data = [json.loads(line) for line in f]

counterfactuals = [d for d in data if d.get("is_counterfactual", False)]
cleans = {d["id"]: d for d in data if not d.get("is_counterfactual", False)}
print(f"Loaded {len(cleans)} clean examples, {len(counterfactuals)} counterfactuals.")

# Build minimal pairs: clean example answered correctly (hop1 label=0) 
# AND its CF twin answered incorrectly (hop1 label=1)
pairs = []
for cf in counterfactuals:
    clean = cleans.get(cf.get("clean_pair_id"))
    if not clean:
        continue
    clean_h1 = next((h for h in clean.get("hops", []) if h["hop_idx"] == 1), None)
    cf_h1 = next((h for h in cf.get("hops", []) if h["hop_idx"] == 1), None)
    if clean_h1 and cf_h1 and clean_h1.get("label") == 0 and cf_h1.get("label") == 1:
        pairs.append((clean, cf))
            

print(f"Found {len(pairs)} perfect minimal pairs.")

In [ ]:
# ===== CELL 4 v3 — logit-based causal tracing =====
# v3: builds the corrupted prompt from the CLEAN prompt (only the question entity differs),
#     removing the spurious system-prompt date difference ('27' -> '30') seen in v2.
# Cells 1-3 unchanged (model, tokenizer, `pairs` must exist).

import re, json, difflib
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

num_layers = model.config.num_hidden_layers
DEV = model.device
WINDOW = 5          # MLP window: layers l-2 .. l+2
MIN_GAP = 1.0       # pair needs >= 1 nat clean-vs-corrupted gap to be a valid test
BATCH = 3           # patched runs per forward pass (3 is safe for T4x2 Kaggle)
RUN_MLP_WINDOW = True

def get_hop1(cot):
    m = re.search(r'(<hop1>.*?</hop1>)', cot, re.DOTALL)
    return m.group(1) if m else ""

def enc(text):
    return tokenizer(text, add_special_tokens=False, return_tensors="pt").input_ids[0]

def changed_blocks(a, b):
    """Token blocks where clean (a) and cf (b) prompts differ. Works even if lengths differ."""
    sm = difflib.SequenceMatcher(a=a.tolist(), b=b.tolist(), autojunk=False)
    return [(i1, i2, j1, j2) for tag, i1, i2, j1, j2 in sm.get_opcodes() if tag != "equal"]

def make_store_hook(store, key):
    def hook(module, args, output):
        h = output[0] if isinstance(output, tuple) else output
        store[key] = h[0].detach().clone()
        return output
    return hook

@torch.no_grad()
def run_batch(input_ids, P, mask, configs):
    """configs: list (one per batch row) of [(module, cf_positions, clean_vectors), ...].
    Returns summed log P(target tokens under mask) for each row."""
    out = []
    tgt = input_ids[P:].to(DEV)
    for s in range(0, len(configs), BATCH):
        chunk = configs[s:s + BATCH]
        by_mod = {}
        for b, cfg in enumerate(chunk):
            for mod, pos, vecs in cfg:
                by_mod.setdefault(mod, []).append((b, pos, vecs))
        handles = []
        for mod, items in by_mod.items():
            def hook(module, args, output, items=items):
                h = output[0] if isinstance(output, tuple) else output
                h = h.clone()
                for b, pos, vecs in items:
                    h[b, pos.to(h.device), :] = vecs.to(h.device)
                return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
            handles.append(mod.register_forward_hook(hook))
        try:
            ids = input_ids.unsqueeze(0).repeat(len(chunk), 1).to(DEV)
            hid = model.model(input_ids=ids, use_cache=False).last_hidden_state[:, P - 1:-1]
            lp = F.log_softmax(model.lm_head(hid).float(), dim=-1)
        finally:
            for hd in handles:
                hd.remove()
        lp = lp.gather(2, tgt.view(1, -1, 1).expand(len(chunk), -1, 1)).squeeze(2)
        out += lp[:, mask.to(DEV)].sum(1).tolist()
    return out

CONDITIONS = ["entity_single", "last_single"] + (["mlp_window"] if RUN_MLP_WINDOW else [])
results = []
skipped = {"no_hop1": 0, "no_diff": 0, "no_gap": 0}
n_len_mismatch_used = 0
n_prompt_fallback = 0

for idx, (clean, cf) in enumerate(tqdm(pairs, desc="Causal tracing")):
    tgt_text = get_hop1(clean["generated_cot"])          # the CORRECT hop 1 from the clean run
    if not tgt_text:
        skipped["no_hop1"] += 1; continue

    # Minimal pair: identical prompt except the question entity (no date / template drift)
    if clean["question"] in clean["prompt"]:
        cf_prompt = clean["prompt"].replace(clean["question"], cf["question"])
    else:
        cf_prompt = cf["prompt"]; n_prompt_fallback += 1
    c_ids, f_ids, t_ids = enc(clean["prompt"]), enc(cf_prompt), enc(tgt_text)
    blocks = changed_blocks(c_ids, f_ids)
    if not blocks:
        skipped["no_diff"] += 1; continue
    n_raw_blocks = len(blocks)
    # Merge into ONE swapped span (the question entity); a title like 'Call The X' vs 'Y The Z'
    # can otherwise be split around a shared token like ' The'.
    blocks = [(min(b[0] for b in blocks), max(b[1] for b in blocks),
               min(b[2] for b in blocks), max(b[3] for b in blocks))]
    if blocks[0][1] == blocks[0][0] or blocks[0][3] == blocks[0][2]:
        skipped["no_diff"] += 1; continue
    if len(c_ids) != len(f_ids):
        n_len_mismatch_used += 1

    if idx < 3:
        for i1, i2, j1, j2 in blocks:
            print(f"[pair {idx}] swapped: '{tokenizer.decode(c_ids[i1:i2])}' -> '{tokenizer.decode(f_ids[j1:j2])}'")

    # Patch site = LAST token of each swapped block (Meng et al.'s "last subject token")
    c_pos = torch.tensor([i2 - 1 for i1, i2, j1, j2 in blocks], device=DEV)
    f_pos = torch.tensor([j2 - 1 for i1, i2, j1, j2 in blocks], device=DEV)

    # Score only reasoning tokens of hop 1, not the copied entity-name tokens
    ent_tok = torch.cat([torch.cat([c_ids[i1:i2], f_ids[j1:j2]]) for i1, i2, j1, j2 in blocks])
    mask = ~torch.isin(t_ids, ent_tok)
    if mask.sum() == 0:
        skipped["no_gap"] += 1; continue

    Pc, Pf = len(c_ids), len(f_ids)
    clean_in, cf_in = torch.cat([c_ids, t_ids]), torch.cat([f_ids, t_ids])

    # 1) clean run: harvest residual stream + MLP outputs at every layer
    store = {}
    hs = []
    for l in range(num_layers):
        hs.append(model.model.layers[l].register_forward_hook(make_store_hook(store, ("res", l))))
        hs.append(model.model.layers[l].mlp.register_forward_hook(make_store_hook(store, ("mlp", l))))
    try:
        lp_clean = run_batch(clean_in, Pc, mask, [[]])[0]
    finally:
        for h in hs:
            h.remove()

    # 2) corrupted run, unpatched
    lp_cf = run_batch(cf_in, Pf, mask, [[]])[0]
    gap = lp_clean - lp_cf
    if gap < MIN_GAP:
        skipped["no_gap"] += 1; continue

    # 3) all patched runs for this pair
    last_c = torch.tensor([Pc - 1], device=DEV)
    last_f = torch.tensor([Pf - 1], device=DEV)
    configs, keys = [], []
    for l in range(num_layers):
        L = model.model.layers
        configs.append([(L[l], f_pos, store[("res", l)][c_pos.to(store[("res", l)].device)])]); keys.append(("entity_single", l))
        configs.append([(L[l], last_f, store[("res", l)][last_c.to(store[("res", l)].device)])]); keys.append(("last_single", l))
        if RUN_MLP_WINDOW:
            win = range(max(0, l - WINDOW // 2), min(num_layers, l + WINDOW // 2 + 1))
            configs.append([(L[k].mlp, f_pos, store[("mlp", k)][c_pos.to(store[("mlp", k)].device)]) for k in win]); keys.append(("mlp_window", l))
    scores = run_batch(cf_in, Pf, mask, configs)

    rec = {k: [0.0] * num_layers for k in CONDITIONS}
    for (k, l), lp in zip(keys, scores):
        rec[k][l] = (lp - lp_cf) / gap              # 0 = no effect, 1 = fully restored
    rec["gap"] = gap
    rec["n_blocks"] = n_raw_blocks
    results.append(rec)
    del store
    torch.cuda.empty_cache()

# ===== Results =====
n = len(results)
print(f"\nPairs: {len(pairs)} | valid: {n} (of which {n_len_mismatch_used} had different prompt lengths) | skipped: {skipped}")
print(f"Prompt fallback (question not found verbatim in clean prompt): {n_prompt_fallback}")
print(f"Pairs whose diff had >1 raw block (merged into one span): {sum(r['n_blocks'] > 1 for r in results)}\n")
assert n > 0, "No valid pairs — check the skipped counts above."
R = {k: np.array([r[k] for r in results]) for k in CONDITIONS}

for k in CONDITIONS:
    mean = R[k].mean(0); ci = 1.96 * R[k].std(0) / np.sqrt(n); frac = (R[k] > 0.5).mean(0) * 100
    print(f"--- {k} ---")
    for l in range(num_layers):
        print(f"Layer {l:02d}: recovery {mean[l]:5.2f} ± {ci[l]:.2f} | pairs >50% restored: {frac[l]:5.1f}%")
    print()

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
for k in CONDITIONS:
    m = R[k].mean(0); c = 1.96 * R[k].std(0) / np.sqrt(n)
    plt.plot(range(num_layers), m, marker="o", label=k)
    plt.fill_between(range(num_layers), m - c, m + c, alpha=0.2)
plt.axhline(0, color="gray", lw=0.8)
plt.xlabel("Layer"); plt.ylabel("Fraction of clean-vs-corrupted gap recovered")
plt.title(f"Causal tracing of hop 1 (n={n} pairs)")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig("causal_trace.png", dpi=200, bbox_inches="tight"); plt.show()

with open("causal_trace_results.json", "w") as f:
    json.dump({"skipped": skipped, "n": n, "n_len_mismatch_used": n_len_mismatch_used,
               "results": [{k: (v if k in ("gap", "n_blocks") else list(map(float, v))) for k, v in r.items()} for r in results]}, f)
print("Saved causal_trace_results.json and causal_trace.png")


In [ ]:
# ===== CELL 6 — Does the probe read the model's internal state, or just the words? =====
# Needs cells 1-3 (model, tokenizer, pairs). Independent of cells 4-5.
#
# Design: the hop-1 TEXT is held fixed (the clean run's correct hop 1) in every condition.
#   clean prompt     + clean hop 1   -> hop is grounded            -> probe should say "success"
#   corrupted prompt + same hop 1    -> hop doesn't fit the question -> a probe reading internal state should rise
#   corrupted + entity patch at L    -> internal state restored     -> probe should drop again
# Because the words never change, any movement in the probe comes from the internal state.
# Features are built exactly as in training (generate_cot.py): layer-L output, mean over the
# hop-1 inner-text tokens, default tokenizer special tokens.

import re, difflib, json, joblib
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from scipy.stats import spearmanr

PROBE_LAYERS = [10, 13]                  # 10 = headline probe; 13 = end of the probe plateau
MIN_GAP = 1.0                            # same validity filter as the causal-tracing cell
BATCH = 3                                # lower to 2 on OOM
CONTROL_LAYERS = list(range(0, 28, 3))   # shuffled-donor control at these layers (saves time)
REPO = "AnishRacherla/LinguaFranca-Phase3"
num_layers = model.config.num_hidden_layers
DEV = model.device
HOP1_RE = re.compile(r"<hop1>(.*?)</hop1>", re.DOTALL | re.IGNORECASE)

probes = {P: joblib.load(hf_hub_download(REPO, f"probes/hop1_probe_layer{P}.joblib", repo_type="dataset"))
          for P in PROBE_LAYERS}

def probe_logit(P, vec):
    """Probe log-odds of FAILURE (class 1). Higher = more confident the hop failed."""
    return float(probes[P].decision_function(np.nan_to_num(vec)[None].astype(np.float64))[0])

def build_inputs(prompt, cot_prefix, c0, c1):
    """Mirror generate_cot.extract_hidden_states tokenization + hop-1 span."""
    full = tokenizer(prompt + cot_prefix, return_tensors="pt").input_ids[0]
    p_len = len(tokenizer(prompt, return_tensors="pt").input_ids[0])
    enc = tokenizer(cot_prefix, return_offsets_mapping=True, add_special_tokens=False)
    if len(full) != p_len + len(enc["input_ids"]):
        return None
    offs = enc["offset_mapping"]
    ts = next((i for i, (cs, ce) in enumerate(offs) if cs <= c0 < ce), None)
    te = next((i for i, (cs, ce) in enumerate(offs) if cs < c1 <= ce), None)
    if ts is None or te is None:
        return None
    return full, p_len, (p_len + ts, p_len + te + 1)

def make_store_hook(store, key):
    def hook(module, args, output):
        h = output[0] if isinstance(output, tuple) else output
        store[key] = h[0].detach().clone()
        return output
    return hook

@torch.no_grad()
def run_rows(input_ids, P, mask, span, configs):
    """Each config = list of (module, positions, vectors) patches for one batch row.
    Returns [(masked target log-prob, {probe_layer: pooled hop-1 vector}), ...]."""
    out = []
    for s in range(0, len(configs), BATCH):
        chunk = configs[s:s + BATCH]
        by_mod = {}
        for b, cfg in enumerate(chunk):
            for mod, pos, vecs in cfg:
                by_mod.setdefault(mod, []).append((b, pos, vecs))
        handles, cap = [], {}
        for mod, items in by_mod.items():                       # patch hooks first ...
            def hook(module, args, output, items=items):
                h = output[0] if isinstance(output, tuple) else output
                h = h.clone()
                for b, pos, vecs in items:
                    h[b, pos.to(h.device), :] = vecs.to(h.device)
                return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
            handles.append(mod.register_forward_hook(hook))
        for Pl in PROBE_LAYERS:                                  # ... then capture (sees patched output)
            def cap_hook(module, args, output, Pl=Pl):
                h = output[0] if isinstance(output, tuple) else output
                cap[Pl] = h[:, span[0]:span[1], :].float().mean(1).cpu().numpy()
            handles.append(model.model.layers[Pl].register_forward_hook(cap_hook))
        try:
            ids = input_ids.unsqueeze(0).repeat(len(chunk), 1).to(DEV)
            hid = model.model(input_ids=ids, use_cache=False).last_hidden_state[:, P - 1:-1]
            lp = F.log_softmax(model.lm_head(hid).float(), dim=-1)
        finally:
            for hd in handles:
                hd.remove()
        tgt = input_ids[P:].to(lp.device)
        lp = lp.gather(2, tgt.view(1, -1, 1).expand(len(chunk), -1, 1)).squeeze(2)
        lps = lp[:, mask.to(lp.device)].sum(1).tolist()
        for b in range(len(chunk)):
            out.append((lps[b], {Pl: cap[Pl][b] for Pl in PROBE_LAYERS}))
    return out

records = []
skipped = {"no_hop1": 0, "tok_mismatch": 0, "no_diff": 0, "no_gap": 0}
donor = None   # previous pair's clean entity vectors -> shuffled control

for idx, (clean, cf) in enumerate(tqdm(pairs, desc="Probe vs patching")):
    cot = clean["generated_cot"]
    m = HOP1_RE.search(cot)
    if not m:
        skipped["no_hop1"] += 1; continue
    cot_prefix = cot[:m.end()]
    cf_prompt = (clean["prompt"].replace(clean["question"], cf["question"])
                 if clean["question"] in clean["prompt"] else cf["prompt"])
    bc = build_inputs(clean["prompt"], cot_prefix, m.start(1), m.end(1))
    bf = build_inputs(cf_prompt, cot_prefix, m.start(1), m.end(1))
    if bc is None or bf is None:
        skipped["tok_mismatch"] += 1; continue
    c_in, Pc, span_c = bc
    f_in, Pf, span_f = bf

    sm = difflib.SequenceMatcher(a=c_in[:Pc].tolist(), b=f_in[:Pf].tolist(), autojunk=False)
    bl = [(i1, i2, j1, j2) for t, i1, i2, j1, j2 in sm.get_opcodes() if t != "equal"]
    if not bl:
        skipped["no_diff"] += 1; continue
    i1, i2 = min(b[0] for b in bl), max(b[1] for b in bl)
    j1, j2 = min(b[2] for b in bl), max(b[3] for b in bl)
    if i2 == i1 or j2 == j1:
        skipped["no_diff"] += 1; continue
    c_pos = torch.tensor([i2 - 1]); f_pos = torch.tensor([j2 - 1])

    t_ids = c_in[Pc:]
    mask = ~torch.isin(t_ids, torch.cat([c_in[i1:i2], f_in[j1:j2]]))
    if mask.sum() == 0:
        skipped["no_gap"] += 1; continue

    store = {}
    hs = [model.model.layers[l].register_forward_hook(make_store_hook(store, l)) for l in range(num_layers)]
    try:
        (lp_clean, feat_clean), = run_rows(c_in, Pc, mask, span_c, [[]])
    finally:
        for h in hs:
            h.remove()
    (lp_cf, feat_cf), = run_rows(f_in, Pf, mask, span_f, [[]])
    gap = lp_clean - lp_cf
    if gap < MIN_GAP:
        skipped["no_gap"] += 1; continue

    Ls = model.model.layers
    configs, keys = [], []
    for l in range(num_layers):
        configs.append([(Ls[l], f_pos, store[l][c_pos.to(store[l].device)])]); keys.append(("entity", l))
    if donor is not None:
        for l in CONTROL_LAYERS:
            configs.append([(Ls[l], f_pos, donor[l])]); keys.append(("control", l))
    res = run_rows(f_in, Pf, mask, span_f, configs)

    rec = {"gap": gap, "recovery": {}, "probe": {P: {"clean": probe_logit(P, feat_clean[P]),
                                                      "cf": probe_logit(P, feat_cf[P]),
                                                      "entity": {}, "control": {}} for P in PROBE_LAYERS}}
    for (kind, l), (lp, feats) in zip(keys, res):
        if kind == "entity":
            rec["recovery"][l] = (lp - lp_cf) / gap
        for P in PROBE_LAYERS:
            rec["probe"][P][kind][l] = probe_logit(P, feats[P])
    records.append(rec)
    donor = {l: store[l][c_pos.to(store[l].device)].clone() for l in range(num_layers)}
    del store
    torch.cuda.empty_cache()

# ===================== Analysis =====================
n = len(records)
print(f"\nPairs: {len(pairs)} | valid: {n} | skipped: {skipped}\n")
assert n > 0
sig = lambda x: 1 / (1 + np.exp(-np.asarray(x)))
rng = np.random.default_rng(0)
def boot(x, B=2000):
    x = np.asarray(x); i = rng.integers(0, len(x), (B, len(x))); mm = x[i].mean(1)
    return np.percentile(mm, 2.5), np.percentile(mm, 97.5)

for P in PROBE_LAYERS:
    cl = np.array([r["probe"][P]["clean"] for r in records])
    co = np.array([r["probe"][P]["cf"] for r in records])
    d = co - cl
    lo, hi = boot(d)
    print(f"================ Probe @ layer {P} ================")
    print("TEST A — same hop-1 words, only the question entity differs:")
    print(f"  P(fail) clean prompt     : mean {sig(cl).mean():.3f}")
    print(f"  P(fail) corrupted prompt : mean {sig(co).mean():.3f}")
    print(f"  log-odds rise (cf - clean): {d.mean():+.2f}  [95% CI {lo:+.2f}, {hi:+.2f}] | "
          f"{(d > 0).mean()*100:.0f}% of pairs rise")

    print(f"TEST B — patch the entity at layer L (only L < {P} can reach the probe's layer):")
    print(f"  {'L':>3} | behaviour recovery | probe log-odds drop | shuffled-control drop")
    xs, ys = [], []
    for l in range(num_layers):
        rec_l = np.array([r["recovery"][l] for r in records])
        drop = np.array([r["probe"][P]["cf"] - r["probe"][P]["entity"][l] for r in records])
        ctrl = [r["probe"][P]["cf"] - r["probe"][P]["control"][l] for r in records if l in r["probe"][P]["control"]]
        cs = f"{np.mean(ctrl):+.2f}" if ctrl else "   -"
        if l < P:
            xs += list(rec_l); ys += list(drop)
        if l <= P + 1:
            print(f"  {l:>3} | {rec_l.mean():18.2f} | {drop.mean():+19.2f} | {cs:>21}")
    rho, pval = spearmanr(xs, ys)
    print(f"  Spearman(behaviour recovery, probe drop) over pairs x layers<{P}: rho = {rho:.2f} (p = {pval:.1e})\n")

with open("probe_causal_results.json", "w") as f:
    json.dump({"skipped": skipped, "n": n, "probe_layers": PROBE_LAYERS, "records": records}, f,
              default=lambda o: float(o))
print("Saved probe_causal_results.json")
